In [ ]:
import json
import pandas as pd

from transformers import AutoTokenizer

MODEL = "Qwen/Qwen3-4B"                     # or "Qwen/Qwen3-8B"
TAG = MODEL.split("/")[-1].lower()          # "qwen3-4b": names every input/output file
tok = AutoTokenizer.from_pretrained(MODEL)
THINK_END = tok.convert_tokens_to_ids("</think>")

with open(f"../data/generations/c1_traces_{TAG}.jsonl") as f:
    recs = [json.loads(line) for line in f]

KEY = ["source", "item", "lang", "cond", "sample"]
ids = {tuple(r[k] for k in KEY): (r["prompt_ids"], r["gen_ids"]) for r in recs}
df = pd.DataFrame(recs).drop(columns=["prompt_ids", "gen_ids"])
del recs

df["cut"] = df.finish == "length"
df["loop"] = (df.rep_trace<0.3) | (df.rep_answer < 0.3)
df["no_close"] = df.truncated & ~df.cut    # stopped (EOS) without ever writing </think>
df["bad"] = df.cut | df.loop | df.no_close

print(df.shape)
print(df.groupby(["cond", "lang"]).bad.mean().unstack().round(2))
ok = df[~df.bad]
print(ok.groupby(["cond", "lang"]).size().unstack())

view generation token by token

In [ ]:
def show_tokens(key, start=0, n=30, where="gen"):
    """Token-by-token view of one generation.
    where="gen": start counts from the first generated token; "all": from the first prompt token.
    pos = absolute position in prompt+gen, the index a teacher-forced forward pass uses."""
    p, g = ids[key]
    seq = p + g
    s = len(p) + start if where == "gen" else start
    rows, prev = [], ""
    for i in range(s, min(s + n, len(seq))):
        text = tok.decode(seq[s:i + 1])
        if text.endswith("\ufffd"):
            piece = "."
        else:
            piece, prev = text[len(prev):], text
        rows.append(dict(pos=i, gen_i=i - len(p), id=seq[i],
                         raw=tok.convert_ids_to_tokens(seq[i]), text=repr(piece)))
    return pd.DataFrame(rows).set_index("pos")

key = ("mclm_math100", 4, "ru", "natural", 0)
# show_tokens(key, start=0, n=10)
show_tokens(key, where="all", n=15)

clean the data ... remove latex and varible names and all

In [ ]:
import re
import unicodedata

CODE = re.compile(r"```.*?```|`[^`\n]*`", re.S)
MATH = re.compile(r"\$\$.*?\$\$|\\\[.*?\\\]|\\\(.*?\\\)|\$[^$\n]*?\$", re.S)
CMD  = re.compile(r"\\[A-Za-z]+")                     # \frac, \boxed, \left ...

def is_letter(c):
    """Unicode letter (L*) or combining mark (M*); marks carry Devanagari vowel signs."""
    return unicodedata.category(c)[0] in "LM"

def clean_line(line):
    words = []
    for w in line.split():
        w = "".join(c if is_letter(c) else " " for c in w)   # non-letters -> space
        for part in w.split():
            if re.fullmatch(r"[A-Za-z]", part):              # lone Latin letter = math variable
                continue
            words.append(part)
    return " ".join(words)

def clean(text):
    text = CODE.sub(" ", text)
    text = MATH.sub(" ", text)
    text = CMD.sub(" ", text)
    return "\n".join(clean_line(l) for l in text.split("\n"))   # keep line structure

ex = df[(df.lang == "hi") & (df.cond == "natural") & (df["item"] == 4) & (df["sample"] == 0)].iloc[0]
print(ex.answer_text[:300]); print("\n==>"); print(clean(ex.answer_text)[:200])

clean every usable generation and compare

In [ ]:
from collections import Counter

SCRIPT_PREFIX = [("LATIN", "Latn"), ("DEVANAGARI", "Deva"), ("CJK", "Hani"), ("HIRAGANA", "Kana"),
                 ("KATAKANA", "Kana"), ("ARABIC", "Arab"), ("CYRILLIC", "Cyrl")]

def script_of(c):
    name = unicodedata.name(c, "")                  # e.g. "DEVANAGARI LETTER NA"
    return next((s for p, s in SCRIPT_PREFIX if name.startswith(p)), "other")

def latin_share(t):
    letters = [c for c in t if unicodedata.category(c)[0] == "L"]   # letters only, no marks
    return sum(c.isascii() for c in letters) / len(letters) if letters else float("nan")

def main_script(t):
    c = Counter(script_of(ch) for ch in t if unicodedata.category(ch)[0] == "L")
    return c.most_common(1)[0][0] if c else "none"

ok = df[~df.bad].copy()
ok["a_clean"] = ok.answer_text.map(clean)
ok["a_lat_raw"] = ok.answer_text.map(latin_share)
ok["a_lat_clean"] = ok.a_clean.map(latin_share)
ok["a_script"] = ok.a_clean.map(main_script)

nat = ok[ok.cond == "natural"]
print(nat.groupby(["lang", "source"])[["a_lat_raw", "a_lat_clean"]].mean().round(2))
print(pd.crosstab([ok.cond, ok.lang], ok.a_script))

let's label the data for which lanaguage it is.... labeller

In [ ]:
import regex     # like re, but supports \p{...} Unicode classes; installed with transformers

def blank(m):
    return " " * len(m.group())                  # same-length replacement keeps offsets

def clean_keep(text):
    """Like clean(), but the result has the same length as text: char i here = char i in text."""
    t = CODE.sub(blank, text)
    t = MATH.sub(blank, t)
    t = CMD.sub(blank, t)
    t = regex.sub(r"[^\p{L}\p{M}\n]", " ", t)                               # non-letters -> space
    t = regex.sub(r"(?<![\p{L}\p{M}])[A-Za-z](?![\p{L}\p{M}])", " ", t)     # lone Latin letter -> space
    return t

SEG = re.compile(r"(?:[^\n]*\S[^\n]*(?:\n|$))+")      # a run of lines that each contain non-space

SCR = {k: regex.compile(p) for k, p in {
    "Latn": r"\p{Latin}", "Deva": r"\p{Devanagari}", "Cyrl": r"\p{Cyrillic}",
    "Arab": r"\p{Arabic}", "Hani": r"\p{Han}", "Kana": r"[\p{Hiragana}\p{Katakana}]"}.items()}

MARK = {k: set(v.split()) for k, v in {
    "ne": "छ छन् छु हो होइन छैन मा पनि गर्न गर्नुहोस् भएको हुन्छ हुने यो त्यो लागि थियो र नै गरेको सक्छ भने अनि",
    "hi": "है हैं था थे में नहीं और यह वह के लिए की का गया करने सकते होता होती हम आप भी",
    "en": "the and of to is we that so this in it for with be are as but let need",
    "fr": "le les de des et est que nous une un dans pour pas sur il qui par avec ce",   # no "on": also English
    "sw": "na ya wa kwa ni za katika kuwa hii hiyo cha vya kama lakini kwamba sisi",     # no "au": also French
    "id": "yang dan untuk dengan ini adalah kita dari akan tidak dalam itu jadi maka atau",
}.items()}
CANDS = {"Deva": ["ne", "hi"], "Latn": ["en", "fr", "sw", "id"]}    # scripts shared by several languages
SINGLE = {"Cyrl": "ru", "Arab": "ar"}
MIN_LETTERS = 15

def label_seg(s):
    counts = {k: len(p.findall(s)) for k, p in SCR.items()}
    n = sum(counts.values())
    cjk = counts["Hani"] + counts["Kana"]
    if n + 2 * cjk < MIN_LETTERS:                # too little text (CJK char ~ a word: weight x3)
        return "-", n
    if cjk > n / 2:
        return ("ja" if counts["Kana"] >= 0.1 * cjk else "zh"), n
    main = max(counts, key=counts.get)
    if main in SINGLE:
        return SINGLE[main], n
    words = s.lower().split()                    # match "Let" as "let"
    hits = {l: sum(w in MARK[l] for w in words) for l in CANDS[main]}
    top = sorted(hits.values())
    if top[-1] == 0 or top[-1] == top[-2]:       # no marker words, or a tie
        return main + "?", n
    return max(hits, key=hits.get), n

def segments(text):
    """[(start_char, end_char, n_letters, label), ...] over paragraphs of text."""
    c = clean_keep(text)
    out = []
    for m in SEG.finditer(c):
        label, n = label_seg(m.group())          # label_seg returns (label, n)
        out.append((m.start(), m.end(), n, label))
    return out
    
r = ok[(ok.lang == "ar") & (ok.cond == "prefill") & (ok["item"] == 27) & (ok["sample"] == 0)].iloc[0]
segs = segments(r.trace_text)
print(len(segs), segs[:3])
print([l for *_, l in segs][:25])

actually label the data

- `t_main / a_main` are for the big picture: what language did the model mostly think in, and what did it answer in. These are the crosstabs.
- `t_labs / t_changes` are for switches: whether the language changed inside the text, and where.
- `t_segs` keeps the character offsets (start, end, n, label), so we can find the exact token where a switch happens (step 5).

In [ ]:
def summarize(segs):
    """Labels in order (no '-'), and the main label weighted by letters."""
    labs = [l for *_, l in segs if l != "-"]
    w = Counter()
    for _, _, n, l in segs:
        if l != "-":
            w[l] += n
    return labs, (w.most_common(1)[0][0] if w else "-")

def n_changes(labs):
    labs = [l for l in labs if not l.endswith("?")]     # ignore unsure paragraphs
    return sum(a != b for a, b in zip(labs, labs[1:]))  # neighbours that differ

ok["t_segs"] = ok.trace_text.map(segments)
ok["a_segs"] = ok.answer_text.map(segments)
ok[["t_labs", "t_main"]] = ok.t_segs.map(summarize).apply(pd.Series)
ok[["a_labs", "a_main"]] = ok.a_segs.map(summarize).apply(pd.Series)
ok["t_changes"] = ok.t_labs.map(n_changes)

print(pd.crosstab([ok.cond, ok.lang], ok.t_main))
print(pd.crosstab([ok.cond, ok.lang], ok.a_main))
print(ok.groupby(["cond", "lang"]).t_changes.apply(lambda s: (s > 0).mean()).unstack().round(2))

find the switch points and sort them

- one row per place where neighbouring (sure) paragraph labels differ: `frm -> to`, char offsets, `frac` = how far through the trace, `run` = how many paragraphs the new label lasts
- `stype`: **final** (change lands on the last paragraph = answer drafted in the answer language before `</think>`), **blip** (run 1, usually quoting the prompt), **short** (run 2), **sustained** (run ≥ 3), **ne<->hi** (Devanagari mixing, not a switch)
- use `x["last"]`, not `x.last`: Series has methods called `.last`, `.item`, `.sample`, `.cat`

In [ ]:
def switch_points(segs):
    """One dict per place where neighbouring (sure) paragraph labels differ."""
    s = [x for x in segs if x[3] != "-" and not x[3].endswith("?")]   # sure labels only
    out = []
    for i in range(1, len(s)):
        if s[i][3] != s[i - 1][3]:
            run = 1                                           # how long the new label lasts
            while i + run < len(s) and s[i + run][3] == s[i][3]:
                run += 1
            out.append(dict(frm=s[i - 1][3], to=s[i][3], prev_start=s[i - 1][0],
                            start=s[i][0], end=s[i][1], run=run, last=(i == len(s) - 1)))
    return out

rows = []
for idx, r in ok[ok.t_changes > 0].iterrows():
    for d in switch_points(r.t_segs):
        rows.append(dict(row=idx, **{k: r[k] for k in KEY},
                         frac=round(d["start"] / len(r.trace_text), 2), **d))
sw = pd.DataFrame(rows)
sw["kind"] = sw["frm"] + "->" + sw["to"]

def switch_type(x):
    if {x["frm"], x["to"]} <= {"ne", "hi"}:
        return "ne<->hi"
    if x["last"]:
        return "final"
    if x["run"] == 1:
        return "blip"
    if x["run"] == 2:
        return "short"
    return "sustained"

sw["stype"] = sw.apply(switch_type, axis=1)
print(sw.shape, "in", sw.row.nunique(), "traces")
print(sw.iloc[0])
print(pd.crosstab([sw["cond"], sw["lang"]], sw["stype"]))
print(sw[(sw["cond"] == "prefill") & (sw["stype"] == "sustained")]["kind"].value_counts())

read switches in context: the paragraph before and the paragraph after each change

In [ ]:
def show(i, w=200):
    x = sw.loc[i]
    t = ok.loc[x["row"]].trace_text
    print(f"--- {x['cond']} {x['lang']} item {x['item']} s{x['sample']}  {x['kind']} {x['stype']} "
          f"run={x['run']} at {x['frac']}")
    print("BEFORE:", t[x["prev_start"]:x["start"]][-w:].strip().replace("\n", " ⏎ "))
    print("AFTER :", t[x["start"]:x["end"]][:w].strip().replace("\n", " ⏎ "))

for stype in ["final", "blip", "sustained"]:
    for i in sw[sw["stype"] == stype].sample(2, random_state=0).index:
        show(i)

positions for every usable trace

In [ ]:
OFFSETS = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]       # tokens before cut
FRACS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]       # fraction of the way to cut

def char_to_tok(trace, c):
    """Index of the token in `trace` that contains character c of tok.decode(trace)."""
    lo, hi = 0, len(trace)                     # smallest k with len(decode(trace[:k+1])) > c
    while lo < hi:
        mid = (lo + hi) // 2
        if len(tok.decode(trace[:mid + 1])) > c:
            hi = mid
        else:
            lo = mid + 1
    return lo

def label_at(segs, c):
    """Label of the paragraph containing character c ('-' if between paragraphs)."""
    for s, e, n, lab in segs:
        if s <= c < e:
            return lab
    return "-"

final_char = sw[sw["stype"] == "final"].set_index("row")["start"]   # ok-index -> char where preview starts

trace_rows, pos_rows = [], []
for idx, r in ok.iterrows():
    key = tuple(r[k] for k in KEY)
    p, g = ids[key]
    te = g.index(THINK_END)                    # </think>, as an index into gen_ids
    trace = g[:te]
    assert tok.decode(trace) == r.trace_text   # char offsets are offsets into exactly this string
    fc = final_char.get(idx)                   # None if no final preview
    cut = char_to_tok(trace, fc) if fc is not None else te
    L = len(p)                                 # absolute position = L + gen index
    trace_rows.append(dict(**{k: r[k] for k in KEY}, prompt_len=L, think_end=L + te, cut=L + cut,
                           has_final=fc is not None, t_main=r.t_main, a_main=r.a_main))

    def add(kind, offset, gi):
        para = label_at(r.t_segs, len(tok.decode(trace[:gi]))) if gi < te else "boundary"
        pos_rows.append(dict(**{k: r[k] for k in KEY}, kind=kind, offset=offset, gen_i=gi, pos=L + gi, para=para))

    for d in OFFSETS:
        if cut - d >= 0:
            add("before_cut", d, cut - d)
    for f in FRACS:
        add("frac", f, int(f * cut))
    add("think_end", 0, te)
    add("answer_start", 1, te + 1)

traces = pd.DataFrame(trace_rows)
pos = pd.DataFrame(pos_rows)
print(traces.shape, pos.shape)
print(traces.has_final.sum(), "traces with a final preview; tokens from cut to </think>:")
print((traces.think_end - traces.cut)[traces.has_final].describe().round(1))

r0 = traces[traces.has_final].iloc[1]          # worked example: decode a few tokens at cut
p, g = ids[tuple(r0[k] for k in KEY)]
print(repr(tok.decode((p + g)[r0.cut:r0.cut + 8])), "...", repr(tok.decode((p + g)[r0.think_end:r0.think_end + 2])))

label each trace's answer and pick the `</think>` test set

In [ ]:
X_LABELS = {"ne": {"ne", "hi", "Deva?"}, "hi": {"hi", "ne", "Deva?"}}   # Devanagari = X for ne/hi

def answer_lang(r):
    if r.a_main == "en":
        return "en"
    if r.a_main in X_LABELS.get(r.lang, {r.lang}):
        return "X"
    return "other"                                   # id, zh for hi, ...

traces["ans"] = traces.apply(answer_lang, axis=1)

nat = traces[(traces.cond == "natural") & (traces.ans != "other")]
per_item = nat.groupby(["lang", "source", "item"]).ans.value_counts().unstack(fill_value=0)
mixed = per_item[(per_item.en >= 1) & (per_item.X >= 1)]                 # items whose samples disagree
print(mixed.groupby(level="lang").agg(items=("en", "size"), en=("en", "sum"), X=("X", "sum")))

traces["think_test"] = traces.set_index(["lang", "source", "item"]).index.isin(mixed.index) \
                       & (traces.cond == "natural") & (traces.ans != "other")
print(traces.think_test.sum(), "traces in the </think> test")

OUT = "../data/activations"
traces.to_csv(f"{OUT}/c1_trace_meta_{TAG}.csv", index=False)
pos.to_csv(f"{OUT}/c1_positions_{TAG}.csv", index=False)